# Õppetund 12 - Vestluse ajaloo vähendamine agentide märkmikuga

See märkmeleht demonstreerib, kuidas hallata konteksti pikkades vestlustes, kasutades Microsoft Agent Frameworki. Kui vestlused kasvavad, suureneb tokenite arv — lõpuks ületades mudeli konteksti akna suuruse. Me lahendame selle **konteksti kokkusurutise mustri** ja **agendi märkmiku** abil salvestava mäluga.

## Mida sa õpid:
1. **Miks konteksti haldamine on oluline**: Mõistmine tokenite piirangutest ja konteksti akendest
2. **Kontekstitundlikud agendid**: Agentide loomine, kes haldavad oma vestluse konteksti
3. **Konteksti kokkusurutise muster**: Tööriistade kasutamine vestluse ajaloo kokkusurumiseks
4. **Agendi märkmik**: Salvestav mälu, mis säilib konteksti vähendamise ajal

## Eeltingimused:
- Azure OpenAI seadistus koos keskkonnamuutujate konfigureerimisega
- Eelnevate õppetundide põhjal agentide mõistete tundmine


## Seadistamine


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Miks konteksti haldamine on oluline

Igal LLM-il on lõplik **konteksti aken** — maksimaalne arv sümboleid, mida ta suudab töödelda ühe päringu jooksul. Mitme vooru vestluse edenedes:

- **Sümbolite arv kasvab lineaariselt** iga kasutajasõnumi ja abi vastuse järel.
- **Päringu sümbolid domineerivad kulu** üle, kuna kogu ajalugu saadetakse iga vooru järel uuesti.
- Lõpuks vestlus **ületab konteksti akna** ning mudel kas lühendab või annab vea.

### Konteksti haldamise strateegiad

| Strateegia | Kuidas see töötab | Kompromiss |
|---|---|---|
| **Lühendamine** | Visatakse välja vanimad sõnumid | Kaob varasem kontekst |
| **Kokkuvõtete tegemine** | Vanemad sõnumid tihendatakse kokkuvõtteks | Mõningane detailikaotus, kuid olulised punktid säilivad |
| **Märkmik / Välismälu** | Võtmefaktid hoitakse vestlusest väljaspool | Vajab tööriistakõnesid, kuid säilib mis tahes vähenduse korral |

Selles sülearvutis kombineerime **kokkuvõtete tegemist** ja **märkmiku tööriista**, et agent suudaks säilitada järjepidevust isegi siis, kui vestluse ajalugu on tihendatud.


## Kontekstitundliku agendi loomine


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## Pika vestluse simuleerimine

Käime läbi mitmekordse vestluse, et näha, kuidas kontekst koguneb. Agent peaks hoidma olulisi detaile (eelistused, eelarve, reisi kuupäevad) kogu vestluse vältel ja näitama jätkusuutlikkust.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Pane tähele, kuidas agent säilitab konteksti varasematest kordadest — ta teab Jaapanist, sushist, templitesse, fotograafiast, 3000-dollarilisest eelarvest, üksiksõidust ja aprillikuust. Lühikeses vestluses toimib see hästi, kuid vestluse kasvades muutub kogu ajaloo uuesti saatmine kalliks.

Jätkame vestlust veel kordustega, et näha konteksti kuhjumist:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Konteksti Kokkuvõtte Musternäide

Vestluse kasvades saame kasutada **kokkuvõtte tööriista**, et kogutud konteksti kokku tõmmata kompaktseks vormiks. Agent kutsub seda tööriista esile, et salvestada olulised eelistused, nii et isegi kui vanemad sõnumid kustutatakse, säilib põhiinfo.

See musternäide on keerukamate ajaloo vähendamise meetodite aluseks:
1. Agent tuvastab vestlusest olulised faktid
2. Kutsub kokkuvõtte tööriista, et need püsivalt salvestada
3. Vanemaid sõnumeid saab turvaliselt eemaldada, sest kokkuvõte haarab olulisima

Allpool määratleme `summarize_preferences` tööriista, mida agent saab kutsuda, et salvestada kompaktne kokkuvõte õpitu kohta.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Kokkuvõte

Selles õppetükis õppisite, kuidas hallata konteksti pikaajalistes agentide vestlustes, kasutades Microsoft Agent Frameworki:

### Põhikontseptsioonid
- **Kontekstaknad on piiratud** — iga vestluse ajaloo token maksab raha ja loeb limiiti.
- **Kokkuvõtte tööriistad** võimaldavad agendil kogunenud konteksti kokku võtta lühikesteks kokkuvõteteks, vähendades tokenite kasutust ja säilitades olulise info.
- **Agendi märkmelehed** pakuvad püsivat välist mälu, mis säilib ka pärast vestluse vähendamist.

### Mida te ehitasite
- **Kontekstitundlik agent**, mis hoiab järjepidevust mitmevoorulistes vestlustes
- **Kokkuvõtte tööriist** (`summarize_preferences`), mis salvestab olulised kasutaja andmed kompaktse vormina
- **Mitmevooruline vestlus**, mis demonstreerib konteksti hoidmist ja muutuste käsitlemist

### Reaalse maailma rakendused
- **Klienditeeninduse robotid**: Mäletavad eelistusi pikkade tugisessioonide vältel
- **Isiklikud assistendid**: Jälgivad käimasolevaid projekte ilma konteksti uuesti selgitamata
- **Hariduslikud juhendajad**: Säilitavad õpilaste edenemise paljude interaktsioonide jooksul

### Järgmised sammud
- Rakenda täielik märkmelehe tööriist failipõhise püsivusega
- Lisa automaatne ajaloo kärpimine pärast kokkuvõtte tegemist
- Ühenda vektorandmebaasidega semantilise mälu otsimiseks
- Loo agendid, kes suudavad vestlusi päevi hiljem täieliku kontekstiga jätkata


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Lahtiütlus**:
See dokument on tõlgitud kasutades AI tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi me püüdleme täpsuse poole, palun pange tähele, et automatiseeritud tõlgetes võib esineda vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlkega seotud eksimustest või valesti mõistmistest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
